In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/credit-card-customer-churn-prediction/Churn_Modelling.csv


In [ ]:
df = pd.read_csv('/kaggle/input/credit-card-customer-churn-prediction/Churn_Modelling.csv')

In [ ]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
df.drop(columns = ['RowNumber','CustomerId','Surname'],inplace=True)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [ ]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
df['Geography'].value_counts()

France     5014
Germany    2509
Spain      2477
Name: Geography, dtype: int64

In [ ]:
df['Gender'].value_counts()

Male      5457
Female    4543
Name: Gender, dtype: int64

In [ ]:
df = pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True)

In [ ]:
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0


In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited'].values

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(11, 11),
            nn.Sigmoid(),
            nn.Linear(11, 11),
            nn.Sigmoid(),
            nn.Linear(11, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = Model()

In [ ]:
print(model)

Model(
  (net): Sequential(
    (0): Linear(in_features=11, out_features=11, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=11, out_features=11, bias=True)
    (3): Sigmoid()
    (4): Linear(in_features=11, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Convert numpy → torch
X_train_tensor = torch.tensor(X_train_trf, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

X_val_size = int(0.2 * len(X_train_tensor))

# manual validation split (same as keras validation_split=0.2)
X_val_tensor = X_train_tensor[:X_val_size]
y_val_tensor = y_train_tensor[:X_val_size]

X_train_tensor = X_train_tensor[X_val_size:]
y_train_tensor = y_train_tensor[X_val_size:]

epochs = 100
batch_size = 50

for epoch in range(epochs):
    model.train()

    for i in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)

    print(f"Epoch {epoch+1}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch 1, Train Loss: 0.4963, Val Loss: 0.5116
Epoch 2, Train Loss: 0.4848, Val Loss: 0.5033
Epoch 3, Train Loss: 0.4759, Val Loss: 0.4957
Epoch 4, Train Loss: 0.4661, Val Loss: 0.4872
Epoch 5, Train Loss: 0.4554, Val Loss: 0.4780
Epoch 6, Train Loss: 0.4443, Val Loss: 0.4687
Epoch 7, Train Loss: 0.4336, Val Loss: 0.4598
Epoch 8, Train Loss: 0.4239, Val Loss: 0.4519
Epoch 9, Train Loss: 0.4158, Val Loss: 0.4455
Epoch 10, Train Loss: 0.4094, Val Loss: 0.4405
Epoch 11, Train Loss: 0.4045, Val Loss: 0.4367
Epoch 12, Train Loss: 0.4007, Val Loss: 0.4340
Epoch 13, Train Loss: 0.3979, Val Loss: 0.4319
Epoch 14, Train Loss: 0.3955, Val Loss: 0.4303
Epoch 15, Train Loss: 0.3935, Val Loss: 0.4289
Epoch 16, Train Loss: 0.3915, Val Loss: 0.4276
Epoch 17, Train Loss: 0.3894, Val Loss: 0.4263
Epoch 18, Train Loss: 0.3873, Val Loss: 0.4250
Epoch 19, Train Loss: 0.3851, Val Loss: 0.4236
Epoch 20, Train Loss: 0.3829, Val Loss: 0.4222
Epoch 21, Train Loss: 0.3806, Val Loss: 0.4208
Epoch 22, Train Loss: 

In [ ]:
X_test_tensor = torch.tensor(X_test_trf, dtype=torch.float32)
y_pred = model(X_test_tensor).detach().numpy()

In [ ]:
y_pred

array([[0.2044038 ],
       [0.29035765],
       [0.21835594],
       ...,
       [0.1132016 ],
       [0.12873812],
       [0.1824502 ]], dtype=float32)

In [ ]:
y_pred = y_pred.argmax(axis=-1)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7975

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

NameError: name 'history' is not defined

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])